In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *

import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:

#df_metadata.head()
def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata = transform_df(df_metadata)
df_metadata['species_id'] = df_metadata['species_id'].astype(str)
df_metadata = df_metadata.set_index('species_id').astype(str)

In [ ]:


def get_inoculumn_sort(x):
    subjects = list(np.sort(x.split('-')[:-1]))
    media =  x.split('-')[-1]
    return '-'.join(subjects + [media])




def get_in(x):
    #print('yay',x)
    if x in list(in_dict.keys()):
     
        return in_dict[x]
    else:
        return ''
#df_info['inoculumn'] = df_info['mesocosm'].transform(lambda x: '-'.join(x.split('-')[1:-1]))
e003_metadata = pd.read_csv('e003_metadata_cultures_round2.csv').drop(columns = 'Unnamed: 0')
e003_metadata['type_meso'] = e003_metadata['type_mesocosm']
#e003_metadata['inoculumn'] = e003_metadata['inoculumn'].transform(get_inoculumn_sort)

in_df = e003_metadata.loc[e003_metadata['is_inoculumn'],:].set_index('inoculumn').copy()

in_series = in_df['sample']
in_dict = in_series.to_dict()


In [ ]:
def analyze_diversity(diversity_df1,minor_strain, minor_strain_subject, separate_plots_per_mesocosm=True):
  #  minor_strain = 

   # p = bokeh.plotting.figure(width = 600, height = 400)

   # if separate_plots_per_mesocosm:
    #plots = []
    p2 = bokeh.plotting.figure(width = 500, height = 300)
    palette = bokeh.palettes.Set2[8]
    for i, type_meso in enumerate(diversity_df1['type_meso'].unique()):
        mesos = diversity_df1.loc[diversity_df1['type_meso'] ==type_meso, 'mesocosm'].unique()
        
     #   p3 = bokeh.plotting.figure(width = 300, height = 200)
        df_type_meso = diversity_df1.loc[diversity_df1['type_meso'] == type_meso,:]
        color = palette[0]
        if type_meso.split('-')[-1] == 'mBHI':
            color = palette[1]
        p2.circle(df_type_meso['passage'].values,
                   df_type_meso[minor_strain].values,color = color, legend_label = type_meso, size = 5)
        for mesocosm in mesos:
            print(mesocosm)
            
            df_meso = diversity_df1.loc[diversity_df1['mesocosm'] == mesocosm,:]
            in_sample = df_meso['inoculumn_sample'].unique()[0]
            #print(type_meso, in_sample)
            if in_sample in df_info.index.values:
                df_meso.loc[in_sample,:] = df_info.loc[in_sample,:]
           # df_meso = pd.concat([df_meso, diversity_df1.loc[diversity_df1['sample'] == in_sample,:]])
            df_meso = df_meso.sort_values(by = 'passage')
          #  p.line(df_meso['passage'].values,
           #        df_meso[minor_strain].values, legend_label)

                #print(np.max(df_meso['passage'].values))
               # print(len(df_meso))
                
            p2.line(df_meso['passage'].values,
                   df_meso[minor_strain].values,color=color)
                

              
    p2.yaxis.axis_label = 'Minor strain abundance'
    p2.xaxis.axis_label = 'Passage'
    inoculumn_stuff = '-'.join(type_meso.split('-')[:-1])
    p2.title.text = f'{minor_strain_subject} in {inoculumn_stuff}'
    p2.y_range = bokeh.models.Range1d(-.005,1.005)
    p2.x_range = bokeh.models.Range1d(-.5,7.5)

      #  plots.append(p2)

            

  #  if separate_plots_per_mesocosm:
    return p2
        
    

In [ ]:
def analyze_fitness(diversity_df1,minor_strain, major_strain,
                               minor_strain_subject, major_strain_subject):
    new_df = []
    for i, type_meso in enumerate(diversity_df1['type_meso'].unique()):
        mesos = diversity_df1.loc[diversity_df1['type_meso'] ==type_meso, 'mesocosm'].unique()
        df_type_meso = diversity_df1.loc[diversity_df1['type_meso'] == type_meso,:]
        for mesocosm in mesos:
            
            df_meso = diversity_df1.loc[diversity_df1['mesocosm'] == mesocosm,:]
            in_sample = df_meso['inoculumn_sample'].unique()[0]
            df_meso['shift_from_inoculumn'] = np.nan
           # df_meso[f'shift {minor_strain_subject}'] = np.nan
            df_meso[f'opp_strain_shift_from_inoculumn'] = np.nan
            if in_sample in df_info.index.values:
                df_meso.loc[in_sample,:] = df_info.loc[in_sample,:]
                df_meso['shift_from_inoculumn'] = df_meso[minor_strain] - df_info.loc[in_sample,minor_strain]
            #    df_meso[f'shift {minor_strain_subject}'] = df_meso[minor_strain] - df_info.loc[in_sample,minor_strain]
                df_meso[f'opp_strain_shift_from_inoculumn'] = df_meso[major_strain] - df_info.loc[in_sample,major_strain]
                
        
            df_meso = df_meso.sort_values(by = 'passage')
            new_df.append(df_meso)
            
    
    return pd.concat(new_df)


In [ ]:
species_fnames = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2/*')
species_names = [fname.split('/')[-1] for fname in species_fnames]

species_names 

In [ ]:
df_full = pd.concat(all_dfs)
df_full['total_shift'] = np.abs(df_full['shift_from_inoculumn'] + df_full['opp_strain_shift_from_inoculumn'])
#df_full = df_full.loc[df_full['total_shift'] < .1,:]

df_full_p5p7 = df_full.loc[df_full['passage'] > 3,:].sort_values(by = 'sample')
df_full_p5p7 = df_full_p5p7.loc[df_full_p5p7['total_shift'] < .1,:]
df_full_p5p7['dir_of_shift'] = 1*(df_full_p5p7['shift_from_inoculumn'] > 0.) + -1*(df_full_p5p7['shift_from_inoculumn'] < 0.)
df_full_p5p7['counts'] = 1. 
df_full_p5p7['zero_change'] = df_full_p5p7['shift_from_inoculumn'] == 0.
df_full_p5p7_gr = df_full_p5p7.groupby(['parent_subjects', 'parent_media', 'media', 'mesocosm', 'inoculumn', 'type_meso', 'inoculumn_sample']).sum().reset_index()
#df_full_p5p7_gr_non_zero = df_full_p5p7_gr.loc[df_full_p5p7_gr['zero_change'] == 0,:]

df_full_p5p7_gr['dir_of_shift'] = 1*(df_full_p5p7_gr['dir_of_shift'] > 0.) + -1*(df_full_p5p7_gr['dir_of_shift'] < 0.)
df_full_p5p7_gr['counts'] = 1
grouped_by_meso = df_full_p5p7_gr.groupby(by=['type_meso', ]).sum()
#(len(grouped_by_meso.loc[grouped_by_meso['dir_of_shift'] == -2,:]) + len(grouped_by_meso.loc[grouped_by_meso['dir_of_shift'] == 2,:])) \
 #   /len(grouped_by_meso.loc[grouped_by_meso['counts'] == 2,:])

grouped_by_meso

In [ ]:
df_full.loc[df_full['total_shift'] > .1,:]

In [ ]:
df_full = pd.concat(all_dfs)
df_full['total_shift'] = np.abs(df_full['shift_from_inoculumn'] + df_full['opp_strain_shift_from_inoculumn']) #/np.abs(df_full['shift_from_inoculumn'])
#df_full_p5p7 = df_full.loc[df_full['passage'] > 3,:].sort_values(by = 'sample')
p = iqplot.ecdf(data = df_full.loc[df_full['passage'] > 3,:], q = 'total_shift')
bokeh.io.show(p)

In [ ]:
e003_metadata

In [ ]:
e003_metadata['type_mesocosm'].unique()

In [ ]:
species = '100099'

fnames = glob(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2/{species}/*_parent_freqs.csv')
   
plots_all = []
all_dfs = []
for info_fname in np.sort(fnames): 
    
       # print(info_fname)
       # fname = f'~/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/AA-AE-mGAM_parent_freqs.csv'
    df = pd.read_csv(info_fname).rename(columns = {'Unnamed: 0': 'sample'})
   # print(df)
       # print(df)
    if len(df) ==0:
        continue

    medium_df = df.set_index('sample').median()
    minor_strain = medium_df.index.values[0]
    major_strain = medium_df.index.values[1]
       # strains = df.columns.values[1:]
      #  subjects = [e003_metadata.loc[e003_metadata['sample'] == strain,'parent_subjects'].values[0].split('-')[0] for strain in strains]
       # print(subjects)

    df_info = pd.concat([df.set_index('sample'), 
                             e003_metadata.loc[e003_metadata['sample'].isin(df['sample'].unique()),:].set_index('sample')],axis=1)
    
    df_info['inoculumn_sample'] = df_info['inoculumn'].transform(get_in) 
        #print(minor_strain)
    minor_strain_subject =  e003_metadata.loc[e003_metadata['sample'] == minor_strain,
                                              'parent_subjects'].values[0].split('-')[0]
    major_strain_subject =  e003_metadata.loc[e003_metadata['sample'] == major_strain,
                                              'parent_subjects'].values[0].split('-')[0]
   # print(minor_strain_subject)
       # print(minor_strain)
    p2 = analyze_diversity(df_info, minor_strain, 
                               e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0])
    df2 = analyze_fitness(df_info, minor_strain, major_strain,
                               minor_strain_subject, major_strain_subject)
    #print(df2)
    #print(df2.head())
  # print(df2.head())
    all_dfs.append(df2[['comm', 'parent_subjects', 'is_inoculumn',
       'parent_media', 'media', 'passage', 'mesocosm',
       'inoculumn', 'type_meso', 'inoculumn_sample', 'opp_strain_shift_from_inoculumn',
       'shift_from_inoculumn']].reset_index())
    bokeh.io.show(p2)
    plots_all.append(p2)

In [ ]:
#species_names = glob.glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snps/*')
#species = '102320'
fnames = glob(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/*_parent_freqs.csv')

for species in species_names:
    fnames = glob(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/*_parent_freqs.csv')
    print(species)
    plots_all = []
    for info_fname in np.sort(fnames): 
       # print(info_fname)
       # fname = f'~/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/AA-AE-mGAM_parent_freqs.csv'
        df = pd.read_csv(info_fname).rename(columns = {'Unnamed: 0': 'sample'})
       # print(df)
        if len(df) ==0:
            continue

        medium_df = df.set_index('sample').median()
        minor_strain = medium_df.index.values[0]
       # strains = df.columns.values[1:]
      #  subjects = [e003_metadata.loc[e003_metadata['sample'] == strain,'parent_subjects'].values[0].split('-')[0] for strain in strains]
       # print(subjects)

        df_info = pd.concat([df.set_index('sample'), 
                             e003_metadata.loc[e003_metadata['sample'].isin(df['sample'].unique()),:].set_index('sample')],axis=1)
        df_info['inoculumn_sample'] = df_info['inoculumn'].transform(get_in)
        #print(minor_strain)
        minor_strain_subject =  e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0]
        print(minor_strain_subject)
       # print(minor_strain)
        p2 = analyze_diversity(df_info, minor_strain, 
                               e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0])
        bokeh.io.show(p2)
        plots_all.append(p2)
    if len(plots_all)>0:
        bokeh.io.export_png(bokeh.layouts.gridplot(plots_all,ncols=2),filename=f'{species}_minor_strain_abundance.png')

In [ ]:
species = '102528'
fnames = glob(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/*_parent_freqs.csv')
plots_all = []
for info_fname in np.sort(fnames): 
    print(info_fname)
   # fname = f'~/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/AA-AE-mGAM_parent_freqs.csv'
    df = pd.read_csv(info_fname).rename(columns = {'Unnamed: 0': 'sample'})

    medium_df = df.set_index('sample').median()
    if len(medium_df) < 1:
        continue
    minor_strain = medium_df.index.values[0]
   # strains = df.columns.values[1:]
  #  subjects = [e003_metadata.loc[e003_metadata['sample'] == strain,'parent_subjects'].values[0].split('-')[0] for strain in strains]
   # print(subjects)
    
    df_info = pd.concat([df.set_index('sample'), e003_metadata.loc[e003_metadata['sample'].isin(df['sample'].unique()),:].set_index('sample')],axis=1)
    df_info['inoculumn_sample'] = df_info['inoculumn'].transform(get_in)
    #print(minor_strain)
    minor_strain_subject =  e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0]
   # print(minor_strain_subject)
    p2 = analyze_diversity(df_info, minor_strain, 
                           e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0])
    bokeh.io.show(p2)
    plots_all.append(p2)
bokeh.io.export_png(bokeh.layouts.gridplot(plots_all,ncols=2),filename=f'{species}_minor_strain_abundance.png')

In [ ]:
species = '100196'
fnames = glob(f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/*_parent_freqs.csv')
plots_all = []
for info_fname in np.sort(fnames): 
    print(info_fname)
   # fname = f'~/git/coalescence-pilot-mgx/workflow/report/track_snps/{species}/AA-AE-mGAM_parent_freqs.csv'
    df = pd.read_csv(info_fname).rename(columns = {'Unnamed: 0': 'sample'})

    medium_df = df.set_index('sample').median()
    if len(medium_df) < 1:
        continue
    minor_strain = medium_df.index.values[0]
   # strains = df.columns.values[1:]
  #  subjects = [e003_metadata.loc[e003_metadata['sample'] == strain,'parent_subjects'].values[0].split('-')[0] for strain in strains]
   # print(subjects)
    
    df_info = pd.concat([df.set_index('sample'), e003_metadata.loc[e003_metadata['sample'].isin(df['sample'].unique()),:].set_index('sample')],axis=1)
    df_info['inoculumn_sample'] = df_info['inoculumn'].transform(get_in)
    #print(minor_strain)
    minor_strain_subject =  e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0]
   # print(minor_strain_subject)
    p2 = analyze_diversity(df_info, minor_strain, 
                           e003_metadata.loc[e003_metadata['sample'] == minor_strain,'parent_subjects'].values[0].split('-')[0])
    bokeh.io.show(p2)
    plots_all.append(p2)
bokeh.io.export_png(bokeh.layouts.gridplot(plots_all,ncols=2),filename=f'{species}_minor_strain_abundance.png')